In [22]:
from pathlib import Path
import numpy as np
import pandas as pd

In [23]:
pd.set_option("display.max_columns", 100)
np.random.seed(42)

In [24]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "cleaned_project_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Project root not found (missing data/cleaned_project_dataset.csv)")

ROOT = find_project_root(Path.cwd())
DATA = ROOT / "data"

# Load ml_df from n3 output (required: run n3_feature_engineering first)
ML_DF_PATH = DATA / "ml_df.csv"
PLAYERS_PATH = DATA / "players.csv"

In [ ]:
import subprocess
import sys

if not ML_DF_PATH.exists():
    print("ml_df.csv not found. Running n3_feature_engineering to generate it...")
    n3_path = ROOT / "analysis" / "n3_feature_engineering.ipynb"
    if n3_path.exists():
        subprocess.run(
            [sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
             "--ExecutePreprocessor.timeout=300", "--output", "n3_executed.ipynb",
             str(n3_path)],
            cwd=str(ROOT / "analysis"),
            check=True,
        )
        print("n3 completed. ml_df.csv should now exist.")
    else:
        raise FileNotFoundError(f"Run n3_feature_engineering.ipynb first to create {ML_DF_PATH}")
else:
    print("ml_df.csv found.")

ml_df.csv found.


In [26]:
ml_df = pd.read_csv(ML_DF_PATH)
if "date" in ml_df.columns:
    ml_df["date"] = pd.to_datetime(ml_df["date"], errors="coerce")

print("ml_df shape:", ml_df.shape)
ml_df[["year", "date", "home_team", "away_team", "result_target"]].head()

ml_df shape: (862, 176)


,year,date,home_team,away_team,result_target
0,1930,1930-07-13,France,Mexico,HomeWin
1,1930,1930-07-13,United States,Belgium,HomeWin
2,1930,1930-07-14,Romania,Peru,HomeWin
3,1930,1930-07-14,Yugoslavia,Brazil,HomeWin
4,1930,1930-07-15,Argentina,France,HomeWin


## Player Features (no leakage)

We only use historical data: cumulative WC goals/scorers BEFORE each match.

In [27]:
def parse_match_name(mn: str) -> tuple[str, str]:
    """Parse 'France vs Mexico' or 'France v Mexico' into (home_team, away_team).
    players.csv uses ' vs ', cleaned_project_dataset uses ' v '."""
    if pd.isna(mn) or not str(mn).strip():
        return ("", "")
    s = str(mn).strip()
    for sep in [" vs ", " v "]:
        if sep in s:
            parts = s.split(sep, 1)
            if len(parts) == 2:
                return (parts[0].strip(), parts[1].strip())
    return ("", "")


# Same team name mapping as n3 (align with cleaned_project_dataset)
TEAM_NAME_MAP = {
    "Korea Republic": "South Korea",
    "IR Iran": "Iran",
    "Côte d'Ivoire": "Ivory Coast",
    "Cote d'Ivoire": "Ivory Coast",
    "USA": "United States",
}


def normalize_team(name: str) -> str:
    """Apply team name mapping for consistency."""
    if pd.isna(name) or not str(name).strip():
        return ""
    return TEAM_NAME_MAP.get(str(name).strip(), str(name).strip())

In [28]:
# --- Load and filter players ---
# We only use historical data to avoid leaking match outcome into features
players = pd.read_csv(PLAYERS_PATH)

# Filter to FIFA Men's World Cup only
wc_mask = players["tournament_name"].astype(str).str.contains("World Cup", case=False, na=False)
players = players[wc_mask].copy()

# Parse match_name to get home_team and away_team for each goal's match
parsed = players["match_name"].apply(parse_match_name)
players["match_home"] = [p[0] for p in parsed]
players["match_away"] = [p[1] for p in parsed]

# Normalize team names
players["match_home"] = players["match_home"].apply(normalize_team)
players["match_away"] = players["match_away"].apply(normalize_team)

# Scoring team: home_team=1 means home scored, away_team=1 means away scored
players["scoring_team"] = np.where(
    players["home_team"] == 1,
    players["match_home"],
    players["match_away"],
)

print("Players (WC only) shape:", players.shape)
players[["year", "match_name", "match_home", "match_away", "scoring_team", "penalty"]].head(10)

Players (WC only) shape: (3637, 24)


,year,match_name,match_home,match_away,scoring_team,penalty
0,1930,France vs Mexico,France,Mexico,France,0
1,1930,France vs Mexico,France,Mexico,France,0
2,1930,France vs Mexico,France,Mexico,France,0
3,1930,France vs Mexico,France,Mexico,Mexico,0
4,1930,France vs Mexico,France,Mexico,France,0
5,1930,United States vs Belgium,United States,Belgium,United States,0
6,1930,United States vs Belgium,United States,Belgium,United States,0
7,1930,United States vs Belgium,United States,Belgium,United States,0
8,1930,Yugoslavia vs Brazil,Yugoslavia,Brazil,Yugoslavia,0
9,1930,Yugoslavia vs Brazil,Yugoslavia,Brazil,Yugoslavia,0


In [29]:
# --- Build match key: (year, home_team, away_team) for joining ---
# Normalize team names in ml_df for consistent join with players
ml_df["home_team_norm"] = ml_df["home_team"].apply(normalize_team)
ml_df["away_team_norm"] = ml_df["away_team"].apply(normalize_team)
ml_df["match_key"] = list(zip(ml_df["year"], ml_df["home_team_norm"], ml_df["away_team_norm"]))
players["match_key"] = list(zip(players["year"], players["match_home"], players["match_away"]))

# Goals per match from players: home_team scored when scoring_team == match_home
home_goals_ser = players.groupby("match_key").apply(lambda g: (g["scoring_team"] == g["match_home"]).sum())
away_goals_ser = players.groupby("match_key").apply(lambda g: (g["scoring_team"] == g["match_away"]).sum())
match_goals = pd.DataFrame({"home_goals_scored": home_goals_ser, "away_goals_scored": away_goals_ser})

# Unique scorers per match per team (count of distinct player_ids who scored)
home_scorers = players[players["scoring_team"] == players["match_home"]].groupby("match_key")["player_id"].nunique()
away_scorers = players[players["scoring_team"] == players["match_away"]].groupby("match_key")["player_id"].nunique()
match_goals["home_unique_scorers"] = home_scorers.reindex(match_goals.index).fillna(0).astype(int)
match_goals["away_unique_scorers"] = away_scorers.reindex(match_goals.index).fillna(0).astype(int)

# Penalty goals per match (exclude own goals from penalty count; penalty=1 means penalty goal)
home_pen = players[(players["scoring_team"] == players["match_home"]) & (players["penalty"] == 1)].groupby("match_key").size()
away_pen = players[(players["scoring_team"] == players["match_away"]) & (players["penalty"] == 1)].groupby("match_key").size()
match_goals["home_penalty_goals"] = home_pen.reindex(match_goals.index).fillna(0).astype(int)
match_goals["away_penalty_goals"] = away_pen.reindex(match_goals.index).fillna(0).astype(int)

match_goals = match_goals.reset_index()
match_goals.head(10)

/var/folders/kk/brw_pvj17nx5_4h1y3t17bkc0000gn/T/ipykernel_59974/2519803299.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  home_goals_ser = players.groupby("match_key").apply(lambda g: (g["scoring_team"] == g["match_home"]).sum())
/var/folders/kk/brw_pvj17nx5_4h1y3t17bkc0000gn/T/ipykernel_59974/2519803299.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  away_goals_ser = players.groupby("match_key").apply(lam

,match_key,home_goals_scored,away_goals_scored,home_unique_scorers,away_unique_scorers,home_penalty_goals,away_penalty_goals
0,"(1930, Argentina, Chile)",3,1,2,1,0,0
1,"(1930, Argentina, France)",1,0,1,0,0,0
2,"(1930, Argentina, Mexico)",6,3,3,2,0,1
3,"(1930, Argentina, United States)",6,1,4,1,0,0
4,"(1930, Brazil, Bolivia)",4,0,2,0,0,0
5,"(1930, Chile, France)",1,0,1,0,0,0
6,"(1930, Chile, Mexico)",3,0,2,0,0,0
7,"(1930, France, Mexico)",4,1,3,1,0,0
8,"(1930, Paraguay, Belgium)",1,0,1,0,0,0
9,"(1930, Romania, Peru)",3,1,3,1,0,0


In [ ]:
# For each match we use only goals from prior WC matches; update cumulative after processing
# Sort ml_df by date for chronological processing
ml_sorted = ml_df.sort_values(["year", "date"]).reset_index(drop=True)

# Build match_key -> (home_goals, away_goals, ...) from players
key_to_goals = match_goals.set_index("match_key").to_dict("index")

# Cumulative goals by team (team -> total goals before current match)
cumulative_goals = {}
cumulative_scorers = {}
cumulative_penalties = {}

home_wc_goals_before = []
away_wc_goals_before = []
home_wc_unique_scorers_before = []
away_wc_unique_scorers_before = []
home_wc_penalty_goals_before = []
away_wc_penalty_goals_before = []

for i, row in ml_sorted.iterrows():
    key = row["match_key"]
    h, a = row["home_team_norm"], row["away_team_norm"]

    # Before this match: use cumulative values
    home_wc_goals_before.append(cumulative_goals.get(h, 0))
    away_wc_goals_before.append(cumulative_goals.get(a, 0))
    home_wc_unique_scorers_before.append(cumulative_scorers.get(h, 0))
    away_wc_unique_scorers_before.append(cumulative_scorers.get(a, 0))
    home_wc_penalty_goals_before.append(cumulative_penalties.get(h, 0))
    away_wc_penalty_goals_before.append(cumulative_penalties.get(a, 0))

    # Update cumulative with this match's goals (from players)
    g = key_to_goals.get(key, {})
    hg = g.get("home_goals_scored", 0)
    ag = g.get("away_goals_scored", 0)
    hs = g.get("home_unique_scorers", 0)
    as_ = g.get("away_unique_scorers", 0)
    hp = g.get("home_penalty_goals", 0)
    ap = g.get("away_penalty_goals", 0)

    cumulative_goals[h] = cumulative_goals.get(h, 0) + hg
    cumulative_goals[a] = cumulative_goals.get(a, 0) + ag
    cumulative_scorers[h] = cumulative_scorers.get(h, 0) + hs
    cumulative_scorers[a] = cumulative_scorers.get(a, 0) + as_
    cumulative_penalties[h] = cumulative_penalties.get(h, 0) + hp
    cumulative_penalties[a] = cumulative_penalties.get(a, 0) + ap

# Add to ml_sorted
ml_sorted["home_wc_goals_before"] = home_wc_goals_before
ml_sorted["away_wc_goals_before"] = away_wc_goals_before
ml_sorted["home_wc_unique_scorers_before"] = home_wc_unique_scorers_before
ml_sorted["away_wc_unique_scorers_before"] = away_wc_unique_scorers_before
ml_sorted["home_wc_penalty_goals_before"] = home_wc_penalty_goals_before
ml_sorted["away_wc_penalty_goals_before"] = away_wc_penalty_goals_before

# Diff feature
ml_sorted["wc_goals_diff_before"] = ml_sorted["home_wc_goals_before"] - ml_sorted["away_wc_goals_before"]

ml_sorted[["year", "home_team", "away_team", "home_wc_goals_before", "away_wc_goals_before", "wc_goals_diff_before"]].head(15)

,year,home_team,away_team,home_wc_goals_before,away_wc_goals_before,wc_goals_diff_before
0,1930,France,Mexico,0,0,0
1,1930,United States,Belgium,0,0,0
2,1930,Romania,Peru,0,0,0
3,1930,Yugoslavia,Brazil,0,0,0
4,1930,Argentina,France,0,4,-4
5,1930,Chile,Mexico,0,1,-1
6,1930,United States,Paraguay,3,0,3
7,1930,Yugoslavia,Bolivia,2,0,2
8,1930,Uruguay,Peru,0,1,-1
9,1930,Argentina,Mexico,1,1,0


In [31]:
# --- Merge player features back into ml_df (preserve original order) ---
# ml_sorted has same match_ids as ml_df, just reordered by date; merge on match_id
player_cols = [
    "home_wc_goals_before", "away_wc_goals_before",
    "home_wc_unique_scorers_before", "away_wc_unique_scorers_before",
    "home_wc_penalty_goals_before", "away_wc_penalty_goals_before",
    "wc_goals_diff_before",
]

merge_df = ml_sorted[["match_id"] + player_cols].copy()
ml_df_v2 = ml_df.merge(merge_df, on="match_id", how="left")

# Drop helper columns used for join
ml_df_v2 = ml_df_v2.drop(columns=["home_team_norm", "away_team_norm", "match_key"], errors="ignore")

# Fill NaN with 0 for matches with no prior player data (e.g. first matches of 1930)
for c in player_cols:
    if c in ml_df_v2.columns:
        ml_df_v2[c] = ml_df_v2[c].fillna(0)

print("ml_df_v2 shape:", ml_df_v2.shape)
ml_df_v2[["year", "home_team", "away_team", "home_wc_goals_before", "away_wc_goals_before", "wc_goals_diff_before"]].head(10)

ml_df_v2 shape: (862, 183)


,year,home_team,away_team,home_wc_goals_before,away_wc_goals_before,wc_goals_diff_before
0,1930,France,Mexico,0,0,0
1,1930,United States,Belgium,0,0,0
2,1930,Romania,Peru,0,0,0
3,1930,Yugoslavia,Brazil,0,0,0
4,1930,Argentina,France,0,4,-4
5,1930,Chile,Mexico,0,1,-1
6,1930,United States,Paraguay,3,0,3
7,1930,Yugoslavia,Bolivia,2,0,2
8,1930,Uruguay,Peru,0,1,-1
9,1930,Argentina,Mexico,1,1,0


In [32]:
# --- Export ml_df_v2 ---
# ml_df_v2 = ml_df + player-derived pre-match features (wc_goals_before, etc.)
output_path = ROOT / "data" / "ml_df_v2.csv"
ml_df_v2.to_csv(output_path, index=False)
print("Saved:", output_path)

Saved: /Users/jeanphilippeauguste/Downloads/DS4-World-Cup-Project-Phase/data/ml_df_v2.csv
